In [ ]:
!pip install datasets[audio]
!pip install gradio
!pip install transformer

In [ ]:
import librosa as lr
from datasets import load_dataset
import gradio as gr
from datasets import Audio


In [ ]:
import numpy as np
import scipy as sp
import matplotlib.pyplot as plt

In [ ]:
# MINDS-14 : Dataset contains recordings of people asking an e-banking system questions in several languages and dialects
# Link to dataset: https://huggingface.co/datasets/PolyAI/minds14

minds = load_dataset("PolyAI/minds14",
                     name="en-AU", # only select the Australian subset
                     split="train")
minds

In [ ]:
# Look at one example
example = minds[0]
example

In [ ]:
# Convert the intent to a meaningful string
id2label = minds.features["intent_class"].int2str
id2label(example["intent_class"])

In [ ]:
# Removal of columns like
# lang_id will have a unique value as we use a subset
# english_transcription same as transcription for the same reason
columns_to_remove = ["lang_id", "english_transcription"]
minds = minds.remove_columns(columns_to_remove)
minds

In [ ]:
# lets listen some music
def generate_audio():
    example = minds.shuffle()[0]
    audio = example["audio"]
    return (
        audio["sampling_rate"],
        audio["array"],
    ), id2label(example["intent_class"])


with gr.Blocks() as demo:
    with gr.Column():
        for _ in range(4):
            audio, label = generate_audio()
            output = gr.Audio(audio, label=label)

demo.launch(debug=True)

In [ ]:
# Plot : Waveform of music
signal = example["audio"]["array"]
sampling_rate = example["audio"]["sampling_rate"]

plt.figure().set_figwidth(12)
lr.display.waveshow(signal, sr=sampling_rate)
plt.show()

PreProcessing audio load_dataset
1. Resampling the audio load_dataset

In [ ]:
# Upsample to 16kHz using cast_column
minds = minds.cast_column("audio", Audio(sampling_rate=16_000))
minds[0]

In [ ]:
# Define a function to filter out samples longer than 20s
MAX_DURATION_IN_SECONDS = 20.0

def is_audio_length_in_range(input_length):
    return input_length < MAX_DURATION_IN_SECONDS

# Then create duration column, filter and remove the column
# use librosa to get example's duration from the audio file
new_column = [lr.get_duration(path=x) for x in minds["path"]]
minds = minds.add_column("duration", new_column)

# use Datasets' `filter` method to apply the filtering function
minds = minds.filter(is_audio_length_in_range, input_columns=["duration"])

# remove the temporary helper column
minds = minds.remove_columns(["duration"])
minds

In [ ]:
# Load pre-trained Whisper checkpoint
from transformers import WhisperFeatureExtractor

feature_extractor = WhisperFeatureExtractor.from_pretrained(
    "openai/whisper-small")

In [ ]:
# Define a function to preprocess a single example
def prepare_dataset(example):
    audio = example["audio"]
    features = feature_extractor(
        audio["array"], sampling_rate=audio["sampling_rate"], padding=True
    )
    return features


# Apply to the dataset using map method
minds = minds.map(prepare_dataset)
minds

In [ ]:
# Visualize the log-mel spectrogram for one example
import numpy as np

example = minds[0]
input_features = example["input_features"]

plt.figure().set_figwidth(12)
librosa.display.specshow(
    np.asarray(input_features[0]),
    x_axis="time",
    y_axis="mel",
    sr=feature_extractor.sampling_rate,
    hop_length=feature_extractor.hop_length,
)
plt.colorbar()
plt.show()

Previously we downloaded to whole dataset onto our local machine then did the preprocessing. what if we do the preprocessing during data streaming without loading complete data onto the our local machine.

Streaming audio data
Cf: https://huggingface.co/docs/datasets/stream

from huggingface_hub import login
login()

In [ ]:
# Enable streaming mode
gigaspeech = load_dataset("speechcolab/gigaspeech", "xs",
                          token=access_token,
                          streaming=True)

In [ ]:
# Read first example (iterate over the dataset)
next(iter(gigaspeech["train"]))

In [ ]:
# Preview some examples (2)
gigaspeech_head = gigaspeech["train"].take(2)
list(gigaspeech_head)